# Reproducing Semenov et al. (2026) from dandiset 001516

Paper. Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler JN, Ibey BL, Kochunov P, Pakhomov AG
(2026). Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed
electric fields. Bioelectrochemistry 171, 109283.
[doi:10.1016/j.bioelechem.2026.109283](https://doi.org/10.1016/j.bioelechem.2026.109283). The claims
file was written from the version of record (main text, Methods, Results, Discussion and the legends
of Figures 1 to 7; the paper names no supplement and links no code).

Dandiset. [DANDI:001516](https://dandiarchive.org/dandiset/001516/0.261006.1833), "Plasma Membrane
Recovery of Cells Following 1 us Electroporation Pulse", version 0.261006.1833. Why this version:
the paper cites no dandiset version (its data statement reads "Data will be made available on
request"), and 0.261006.1833 is the only published version, so it is the most recent one. The draft
has the same 72 assets and size; it was not used, as a draft can change.

Claims. `claims/001516/10.1016-j.bioelechem.2026.109283.yaml`, 32 claims. What the dataset holds
(the data check below): 72 image sets from 12 fields of view recorded on 2023-06-22, all stained
with FluoVolt, with 1.00-mm electrodes, 20 sham frames and 40 pulse frames per set, single pulses or
pulse pairs at 50 to 250 V (described as 0.15 to 1.57 kV/cm) and pair intervals of 1, 5, 25, 125
and 250 us (the description lists four intervals, without 250 us). None of the paper's experiments is among them: Figures 1, 2B to 2D, 4B to 4D, 5 and 7
used Di-8-ANEPPS, Figures 2A, 3 and 4A a FluoVolt escalation from 43 to 1400 V/cm in sets of 40 and
60 images with 1.6-mm electrodes, and Figure 6 Fluo-4 calcium imaging. Only the one claim that is a
computation from Eq. (2), with no recordings, can be tested.

| Claims | Tested here | Outcome |
|---|---|---|
| fig5b-conductivity-calibration (Eq. 2, no recordings) | yes | see its section |
| fig1, fig2b, fig2c, fig2d, fig4b, fig4c, fig4d, fig5a, fig5c (Di-8-ANEPPS, 1-us and 0.3-us escalations, Gd3+) | no | blocked, data missing |
| fig2a, fig3, fig4a (FluoVolt and Di-8-ANEPPS escalations from 43 to 1400 V/cm) | no | blocked, data missing |
| fig6a, fig6b (Fluo-4 calcium imaging) | no | blocked, data missing |
| fig7a, fig7b, fig7ab, fig7c (two-pulse recovery, Di-8-ANEPPS, n = 5) | no | blocked, data missing |

The last section measures the dataset's own FluoVolt sets with the paper's method, to show what they
hold; it decides no claim.

In [ ]:
import re
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pulsed_imp as P

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
meta = P.version_metadata()
assets = P.list_assets()
sets = P.load_all(assets)
sets.sort(key=lambda s: (int(P.fov(s["subject"])[1:]), s["starting_time"]))
P.match_cells(sets)
print(f"loaded {len(sets)} of {len(assets)} files from dandiset {P.DANDISET_ID} version {P.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each file was streamed and reduced as it loaded: per cell found in the field, the 6-pixel-wide
perimeter trace of every frame (360 points). Each check of the metadata against the values is stated
under its output.

In [ ]:
print("Dandiset:", meta["name"])
print("Description:", meta["description"])
print("Related:", [(r["relation"], r.get("identifier")) for r in meta.get("relatedResource", [])])
print("Archive summary: files", meta["assetsSummary"]["numberOfFiles"], "subjects", meta["assetsSummary"]["numberOfSubjects"],
      "species", [s["name"] for s in meta["assetsSummary"]["species"]])
info = pd.DataFrame([{
    "fov": P.fov(s["subject"]), "session_id": s["session_id"], "kind": s["kind"], "V1": s["v1"], "V2": s["v2"],
    "interval_us": s["interval_us"], "frames": s["shape"][0], "height": s["shape"][1], "width": s["shape"][2],
    "dtype": s["dtype"], "unit": s["unit"], "rate_hz": s["rate"], "min": s["min"], "max": s["max"], "n_near_max": s["n_near_max"],
    "cells": len(s["cells"]), "acquisition": ",".join(s["acquisition"]),
    "stimulus": ",".join(s["stimulus"]) or "-", "processing": ",".join(s["processing"]) or "-"} for s in sets])
print("\nAcross all files:")
for col in ["frames", "height", "width", "dtype", "unit", "rate_hz", "session_id", "acquisition", "stimulus", "processing"]:
    print(f"  {col}: {sorted(info[col].astype(str).unique())}")
print(f"  pixel values: {info['min'].min():.0f} to {info['max'].max():.0f} (uint16 counts, unit 'n.a.')")
print(f"  files with pixels at 16000 or more (near the 14-bit ceiling 16383): {(info.n_near_max > 0).sum()}, "
      f"{info.n_near_max.sum()} pixels in all of {len(info) * 60 * 512 * 512}")
print(f"  of them within 1.3 equivalent radii of a measured cell: {sum(c['n_near_max'] for s in sets for c in s['cells'])}")
print("\nFiles per field of view and protocol:")
print(info.groupby(["fov", "kind", "interval_us"], dropna=False).size().unstack(["kind", "interval_us"]).fillna(0).astype(int).to_string())

There is one ImageSeries per file, `StrobeImaging`, 60 frames of 512 x 512 uint16 pixels at 6 Hz,
and nothing else: no stimulus, epoch or trial table, no processing module, no pixel size. The 12
"subjects" are fields of view; the pulse protocol of each set is only in the free text of
`session_description`. One pixel in one file is near the 14-bit ceiling and none on a measured
cell, so saturation does not affect the measures. Every field of view has six sets: a sham and five single pulses (c6, c7),
or six pulse pairs at one interval (two fields per interval), so the description's single and
paired protocols are all present, with 2 fields of view per protocol.

In [ ]:
desc = [s["description"] for s in sets]
print("Dye named in the descriptions:", pd.Series([re.search(r"dye (\S+)", d).group(1) for d in desc]).value_counts().to_dict())
print("Electrode spacing:", pd.Series([re.search(r"Distance between electrodes ([\d.]+ mm)", d).group(1) for d in desc]).value_counts().to_dict())
print("Filter cube:", pd.Series([re.search(r"a (\S+) filter cube", d).group(1) for d in desc]).value_counts().to_dict())
general = sorted(set(d.split("####General experiment description#####")[1] for d in desc))
print("Distinct general descriptions:", len(general))
print(general[0][:900])
print("\nAmplitudes of the first pulse (single pulse for c6, c7) in recording order, and the second pulse:")
for f, g in info.groupby("fov", sort=False):
    print(f"  {f:>4}: {g.kind.iloc[-1]:6} interval {g.interval_us.iloc[-1]} us; V1 {list(g.V1)}; V2 {sorted(set(g.V2.dropna().astype(int)))}")
print("\nField strength assigned to each amplitude (the description's list, in order):",
      {v: P.field_of(v) for v in P.VOLTS})

All 72 files name FluoVolt with a FITC filter cube, and 1.00-mm electrodes. The paper's two-pulse
experiment (Fig. 7) used Di-8-ANEPPS and its electrodes were 1.6 mm apart (Methods 2.2). Within each
field of view the first-pulse amplitude rises from 0 to 250 V across the six sets, except c1, whose
50-V pair was recorded before its sham pair. The test pulse is always 50 V, which the description
gives as 0.15 kV/cm, the paper's 150 V/cm test pulse. The description lists the field strengths 0.15,
0.52, 0.9, 1.27 and 1.57 kV/cm for 50 to 250 V; the files give no mapping, so the list is read in
order. The general description gives the intervals as "1, 5, 25, and 125 us", but fields c5 and
c12 hold pairs at 250 us. The field mapping is not proportional to voltage (5 times the voltage for 10.5 times the field),
while the paper states that the current scaled linearly with voltage (Methods 2.2); it is checked
against the responses below.

In [ ]:
print("Distinct data arrays (md5):", len({s["data_hash"] for s in sets}), "of", len(sets))
small = {}
for s in sets:
    small.setdefault(P.fov(s["subject"]), []).append(s["mean_img_small"].ravel().astype(float))
def zs(v):
    return (v - v.mean()) / v.std()
within = [np.mean(zs(a) * zs(b)) for v in small.values() for a, b in zip(v[:-1], v[1:])]
keys = list(small)
fovmean = {k: zs(np.mean(v, 0)) for k, v in small.items()}
between = [(np.mean(fovmean[a] * fovmean[b]), a, b) for i, a in enumerate(keys) for b in keys[i + 1:]]
print(f"Correlation of mean images, consecutive sets of one field of view: {min(within):.3f} to {max(within):.3f}")
mx = max(between)
print(f"Between fields of view: {min(between)[0]:.2f} to {mx[0]:.2f} (highest {mx[1]} and {mx[2]})")
t0 = pd.DataFrame([{"fov": P.fov(s["subject"]), "session_start": s["session_start"],
                    "starting_time": s["starting_time"],
                    "start_epoch": datetime.fromisoformat(s["session_start"]).timestamp()} for s in sets])
t0["starting_minus_epoch_s"] = t0.starting_time - t0.start_epoch
print("\nImageSeries starting_time:", f"{t0.starting_time.min():.0f} to {t0.starting_time.max():.0f} s after session_start_time")
print("starting_time minus the Unix time of the file's session_start_time (s):",
      sorted(t0.starting_minus_epoch_s.round(0).unique()))
print("Time between consecutive sets of a field of view (s):",
      f"{np.nanmin(t0.groupby('fov').starting_time.diff()):.0f} to {np.nanmax(t0.groupby('fov').starting_time.diff()):.0f}")

No array is deposited twice. Consecutive sets of one field of view are near-identical images (the
same cells), while different fields of view correlate at most moderately (the shared illumination
profile), so the 12 fields of view are 12 distinct recordings. `starting_time` holds the Unix time
of each set's acquisition, the same instant as its `session_start_time`, so in NWB terms every
recording starts about 53 years after its session starts; NWB defines `starting_time` relative to
the session's reference time, so this is a format error. Each file has its own
`session_start_time` (the set's time), although all share the session id `session_2023-06-22`.

In [ ]:
cath = P.cathode_angles(sets)
cells = pd.DataFrame([{"fov": P.fov(s["subject"]), "set": i, "cell": c["cell_id"], "area_px": c["area"],
                       "r_eq_px": c["r_eq"], "ring_r_px": c["ring_r"], "drift_px": c["drift_px"]}
                      for s in sets for i, c in enumerate(s["cells"])])
per = cells.groupby("fov", sort=False).agg(cells_per_set=("cell", lambda x: x.size // 6), r_eq_min=("r_eq_px", "min"),
                                           r_eq_max=("r_eq_px", "max"), max_drift_px=("drift_px", "max"))
per["cells_per_set_all_equal"] = [len(set(g.groupby("set").size())) == 1 for _, g in cells.groupby("fov", sort=False)]
per["cathode_deg"] = [cath["CHO_20230622_" + f] for f in per.index]
print(per.round(1).to_string())
print("Cells in all:", int(per.cells_per_set.sum()), "in", len(per), "fields of view")
print("\nCathode direction (degrees counterclockwise from image +x, image y up), range over fields of view:",
      f"{min(cath.values()):.1f} to {max(cath.values()):.1f}")

Each field of view holds 1 to 4 whole cells, found in every one of its sets, with centroids that
move by a few pixels at most between sets, so each cell is the same cell across its six sets. The
files do not say which cell was the experiment's cell (the paper used one cell per experiment, and
the subject ids, c1 to c12, name a field of view), so the measures below use every whole cell and
break the results down by field of view. Radii are in pixels: the files give no pixel size, so the
radius in um that Eq. (1) needs cannot be computed from them.

The field direction is not recorded. It is taken from the data: the first angular harmonic of dF
points the same way in every field of view, about 80 to 104 degrees (image up), which is then the
cathode-facing pole (the pole that brightens, Methods 2.3).

In [ ]:
scan = []
for s in sets:
    for c in s["cells"]:
        a = P.log_asym(c["trace"], cath[s["subject"]])
        t = [abs(stats.ttest_ind(a[:k], a[k:]).statistic) for k in range(5, 56)]
        label = "sham" if s["kind"] == "sham" else ("pair, 0 V first" if (s["kind"] == "pair" and s["v1"] == 0) else s["kind"])
        scan.append({"fov": P.fov(s["subject"]), "protocol": label, "best_split": 5 + int(np.argmax(t)),
                     "step_at_20": 100 * (a[20:].mean() - a[:20].mean()),
                     "step_within_baseline": 100 * (a[10:20].mean() - a[:10].mean())})
scan = pd.DataFrame(scan)
print("Frame after which the cathode/anode log ratio changes most (frames are numbered from 0; 20 = after the 20 sham frames),")
print("number of cell-sets per protocol:")
print(pd.crosstab(scan.protocol, scan.best_split).to_string())
print("\nStep of the log ratio x 100, frames 20-59 minus 0-19, and the null step within the baseline (frames 10-19 minus 0-9):")
print(scan.groupby("protocol")[["step_at_20", "step_within_baseline"]].agg(["mean", "min", "max"]).round(1).to_string())

The 20 sham frames come first. In the pulse sets the polar contrast changes after frame 20 (frames
counted from 0) in 170 of 174 cell-sets with a pulse, and the step is 8 to 77 (log ratio x 100);
in the sham sets the best split is elsewhere and the step is -4.6 to 0.3, and inside the baseline,
where nothing changes, it is -3.0 to 6.2. This contrast between conditions shows that
the frames and the pulse protocol share one clock, and that the description's 20 sham and 40 pulse
exposures are the order of the frames.

In [ ]:
pt = P.pole_table(sets, cath)
print("Uniform part of the collapsed profile (mean dF over the 360 points, %) by protocol, mean and range over cells:")
pt["protocol"] = np.where(pt.kind == "sham", "sham", np.where((pt.kind == "pair") & (pt.V1 == 0), "pair, 0 V first", pt.kind))
print(pt.groupby("protocol").offset.agg(["mean", "min", "max", "size"]).round(1).to_string())
fm = np.array([s["frame_mean"] for s in sets])
print("\nWhole-image mean of frame 0 relative to frames 1-19, median over files:", f"{np.median(fm[:, 0] / fm[:, 1:20].mean(1)):.3f}")
print("Whole-image mean, frames 20-59 relative to frames 1-19, median (range) over files:",
      f"{np.median(fm[:, 20:].mean(1) / fm[:, 1:20].mean(1)):.3f} ({(fm[:, 20:].mean(1) / fm[:, 1:20].mean(1)).min():.3f} to "
      f"{(fm[:, 20:].mean(1) / fm[:, 1:20].mean(1)).max():.3f})")
print("Frame-to-frame variation of the whole-image mean (s.d. / mean, frames 1-19), median over files:",
      f"{np.median(fm[:, 1:20].std(1) / fm[:, 1:20].mean(1)):.3f}")
sg = pt[pt.kind == "single"].pivot_table(index=["fov", "cell"], columns="V1", values="h1_amp")
ratio = sg[100] / sg[50]
print("\nSingle-pulse cells, first-harmonic amplitude of dF (%) at 50 and 100 V and their ratio:")
print(pd.DataFrame({"50 V": sg[50], "100 V": sg[100], "ratio": ratio}).round(2).to_string())
print(f"Ratio expected if dF scales with voltage: 2.0; with the listed fields: {P.field_of(100) / P.field_of(50):.2f}")
print("First-harmonic amplitude in the sham sets (%):", pt[pt.kind == "sham"].h1_amp.round(2).tolist())

The collapsed profiles carry a uniform offset: on the cells, the last 40 frames are dimmer than the
first 20 by 9.3 to 16.8% in the sham sets, with no pulse, so the offset is not a field response. The
whole image dims less (median ratio 0.97), so the cells' dye fades more than the background,
consistent with bleaching. The first frame of each set is also dimmer (median 0.64 of frames 1 to
19). The paper's measure does not remove the offset, and it shifts both poles equally. The flash-to-
flash variation of the whole image is about 4% (median s.d. over mean), which the paper says
collapsing the frames is meant to average out. The sham sets have almost no first harmonic, as expected without a field.

The response at 100 V is 0.96 to 1.84 times that at 50 V. This is closer to a field proportional to
voltage (2.0) than to the listed fields (3.47), but FluoVolt saturates (Results 3.2), which
compresses the ratio, so the mapping of amplitudes to field strengths cannot be checked from the data
and is used as the description gives it.

Summary of the comparison with the paper (Methods 2.1 to 2.3, Results 3.7, legends of Figs. 2, 4 and 7):

| | Paper | Dataset |
|---|---|---|
| Dye | Di-8-ANEPPS (Figs. 1, 2B to D, 4B to D, 5, 7); FluoVolt (Figs. 2A, 3, 4A) | FluoVolt in all 72 files |
| Electrodes | 1.6 mm apart | 1.00 mm |
| Frames per set | 40 baseline, 60 with pulses | 20 sham, 40 with pulses |
| 1-us fields | 43 to 1400 V/cm (43, 150, 230, 360, 570, 780, 1000, 1200, 1400) | 150, 520, 900, 1270, 1570 V/cm |
| Two-pulse interval | 5 us only | 1, 5, 25, 125 and 250 us |
| Two-pulse cells | n = 5, Di-8-ANEPPS | 2 fields of view per interval (4 cells at 5 us) |
| FluoVolt escalation | n = 4 cells | 2 fields of view, single pulses (c6, c7) |

The paper's Discussion names this dataset's design as future work: "Future studies should focus on
the time course of membrane recovery, perhaps using the same two-pulse protocol as in Fig. 7, but
varying the time interval between the pulses." The dataset does not hold the recordings behind any
of the paper's figures.

## fig5b-conductivity-calibration

> By Eq. (2), the effective membrane conductivity barely affects charging while it stays far below
> cytosol and medium conductivities; a 10-fold rise reduces IMP by about 2% and a 100-fold rise by
> 12%.

Criterion: a computation from Eq. (2) with the constants of Methods 2.3 (lambda_i 0.4 S/m, lambda_o
1.64 S/m, R 8 um, d 5 nm, intact lambda_m 3e-7 S/m); the reduction is relative to IMPC, readings (a)
1 - f/1.5 and (b) 1 - f(lambda_m)/f(3e-7 S/m). Reproduced when both values are within tolerance (2 +-
0.2% and 12 +- 1.2%) under one reading. It uses no recordings.

In [ ]:
lam0 = 3e-7
f0 = P.lambda_f(lam0)
print(f"f at the intact conductivity 3e-7 S/m: {f0:.6f} (1.5 for a non-conducting membrane)")
res = []
for fold in [10, 100]:
    f = P.lambda_f(lam0 * fold)
    res.append({"fold": fold, "lambda_m_S_per_m": f"{lam0 * fold:.0e}", "f": f,
                "reduction_a_pct (1 - f/1.5)": 100 * (1 - f / 1.5),
                "reduction_b_pct (1 - f/f0)": 100 * (1 - f / f0)})
res = pd.DataFrame(res)
print(res.round(4).to_string(index=False))
exp = {10: (2, 0.2), 100: (12, 1.2)}
for reading in ["reduction_a_pct (1 - f/1.5)", "reduction_b_pct (1 - f/f0)"]:
    ok = [abs(r[reading] - exp[r.fold][0]) <= exp[r.fold][1] for _, r in res.iterrows()]
    print(f"{reading}: 10-fold within 2 +- 0.2: {ok[0]}; 100-fold within 12 +- 1.2: {ok[1]}")
folds = np.logspace(0, 3, 3001)
red = 100 * (1 - P.lambda_f(lam0 * folds) / 1.5)
print(f"\nFold rise that gives a 2% reduction (reading a): {folds[np.argmin(abs(red - 2))]:.1f}; "
      f"(reading b): {folds[np.argmin(abs(100 * (1 - P.lambda_f(lam0 * folds) / f0) - 2))]:.1f}")
print("\nVariant (not the criterion): the 10-fold reduction (reading a, %) for other radii and membrane thicknesses")
grid = pd.DataFrame({f"d {d * 1e9:.0f} nm": {f"R {R * 1e6:.0f} um": 100 * (1 - P.lambda_f(lam0 * 10, R=R, d=d) / 1.5)
                                             for R in [5e-6, 6e-6, 8e-6, 10e-6, 12e-6]} for d in [4e-9, 5e-9, 7e-9]})
print(grid.round(2).to_string())
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogx(folds, red)
ax.axhline(2, ls=":", c="gray"); ax.axhline(12, ls=":", c="gray"); ax.axvline(10, ls=":", c="gray"); ax.axvline(100, ls=":", c="gray")
ax.set_xlabel("effective conductivity rise (fold, from 3e-7 S/m)"); ax.set_ylabel("IMP reduction (% of IMPC)")
ax.set_title(f"Eq. (2); y range {red.min():.2f} to {red.max():.1f}%")
plt.show()

**Result: not reproduced.** Eq. (2) with the Methods' constants gives a 1.33% IMP reduction for a
10-fold rise in effective conductivity under reading (a) and 1.20% under reading (b), outside 2 +-
0.2%; a 100-fold rise gives 11.87% and 11.75%, within 12 +- 1.2%. A 2% reduction needs a 15- to
16-fold rise. In the variant grid the 10-fold value is 0.60 to 2.46%, and within 2 +- 0.2% only for a
10-um radius with a 4-nm membrane (2.06%) or a 12-um radius with a 5-nm membrane (1.98%), not with
the stated 8 um and 5 nm; the paper's ~2% does not follow from its stated constants, and it may be a
reading off the curve of Fig. 5B.

## What the dataset holds, measured with the paper's method (no claim is decided here)

These are the dataset's own FluoVolt experiments, which the paper does not report. They are measured
with the paper's collapsed pole measure (Methods 2.3) so that the record shows what the data hold
and how close they come to the paper's Fig. 7 design. They are not the paper's data (dye, field
strengths, electrode spacing and cell numbers differ), so they decide no outcome. Two versions are
shown: the paper's measure (`cath`, `anod`), and the same with the profile's uniform offset removed
(`cath_c`, `anod_c`), since the sham sets show that offset is not a field response. Cells of one
field of view were recorded together, so fields of view, not cells, are the independent experiments.

In [ ]:
sp = pt[pt.fov.isin(["c6", "c7"])]
print("Single 1-us pulses (c6, c7), collapsed pole dF (%), mean and s.e. over the 6 cells:")
print(sp.groupby("E1")[["cath", "anod", "offset", "cath_c", "anod_c"]].agg(["mean", "sem"]).round(1).to_string())
print("\nPer field of view (mean over its cells):")
print(sp.pivot_table(index="E1", columns="fov", values=["cath_c", "anod_c"]).round(1).to_string())
print("\nLargest mean |dF| over field strengths, offset removed: cathodic",
      f"{sp.groupby('E1').cath_c.mean().max():.1f}% at {sp.groupby('E1').cath_c.mean().idxmax():.0f} V/cm; anodic",
      f"{sp.groupby('E1').anod_c.mean().min():.1f}% at {sp.groupby('E1').anod_c.mean().idxmin():.0f} V/cm")
print("Range of every value above:", f"{sp[['cath', 'anod', 'cath_c', 'anod_c']].min().min():.1f} to {sp[['cath', 'anod', 'cath_c', 'anod_c']].max().max():.1f}%")

With single pulses the FluoVolt response at the poles, offset removed, is largest at 150 to 520
V/cm and does not grow at stronger fields: the cathodic response is 14.4 and 14.3% at 150 and 520
V/cm, 7.1 and 8.0% at 900 and 1270 V/cm, and 13.3% at 1570 V/cm; the anodic response peaks at -24.7%
at 520 V/cm and is -18.9 to -20.9% above it. This is the pattern the paper reports for FluoVolt (saturation, then a smaller cathodic
response), at smaller values than the paper's 66 to 71% and -83%; the files give no pixel size and
no calibration, and the trace placement differs from the paper's hand-drawn trace, so the sizes are
not comparable.

In [ ]:
pr = pt[pt.kind == "pair"].copy()
def pct_of_control(df, col):
    t = df.pivot_table(index=["interval_us", "fov", "cell"], columns="E1", values=col)
    return 100 * t.div(t[150.0], axis=0)
print("Pulse pairs, 5-us interval (c2, c9): test-pulse pole dF as % of the 150/150 V/cm set, per cell")
five = pr[pr.interval_us == 5]
for col in ["cath", "anod", "cath_c", "anod_c"]:
    p = pct_of_control(five, col)
    print(f"\n{col}:")
    print(p.round(0).to_string())
    fovm = p.groupby("fov").mean()
    print("mean over cells:", p.mean().round(0).to_dict())
    print("mean over fields of view:", fovm.mean().round(0).to_dict())
print("\nRaw test-pulse pole dF (%) at the 5-us interval, mean over cells:")
print(five.groupby("E1")[["cath", "anod", "offset", "cath_c", "anod_c"]].mean().round(1).to_string())

In the paper's measure the cathodic value at the 150/150 control is small (8.6%) next to the
uniform offset it contains (-7.9%), so its percentages vary widely between cells (33 to 368%). With
the offset removed, the cathodic test-pulse response at the 5-us interval is 91 to 93% of the
control (mean over cells) after conditioning at 520 and 900 V/cm, 76% after 1270 V/cm and 65% after
1570 V/cm (60% as a mean over the two fields of view); the anodic response falls less, to 95% and
91%. In the paper's measure the anodic pole falls to 75% and 62%. This resembles the paper's Fig. 7B (bottom),
in another dye and with two fields of view.

In [ ]:
print("All intervals: test-pulse pole dF with the offset removed, % of the 150/150 set, mean over fields of view (each the mean of its cells)")
for col in ["cath_c", "anod_c"]:
    p = pct_of_control(pr, col)
    print(f"\n{col}:")
    print(p.groupby(["interval_us", "fov"]).mean().groupby("interval_us").mean().round(0).to_string())
print("\nThe same, the paper's measure (no offset removed), cathodic and anodic:")
for col in ["cath", "anod"]:
    p = pct_of_control(pr, col)
    print(f"\n{col}:")
    print(p.groupby(["interval_us", "fov"]).mean().groupby("interval_us").mean().round(0).to_string())
print("\nTest pulse alone (first pulse 0 V) against the 150/150 set, offset removed, mean over fields of view:",
      pct_of_control(pr, "cath_c").groupby(["interval_us", "fov"]).mean()[0.0].groupby("interval_us").mean().round(0).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 3), sharey=True)
ang = np.arange(P.N_ANG) + 0.5
lo, hi = np.inf, -np.inf
profiles = {}
for s in sets:
    if s["kind"] == "pair" and s["interval_us"] == 5:
        for c in s["cells"]:
            df = P.profile_df(c["trace"])
            # rotate so that index k is the angle k + 0.5 degrees from the cathode-facing pole
            profiles.setdefault(s["v1"], []).append(np.roll(df - df.mean(), -int(round(cath[s["subject"]] - 0.5))))
for ax, v in zip(axes, sorted(profiles)):
    m = np.mean(profiles[v], 0)
    lo, hi = min(lo, m.min()), max(hi, m.max())
    ax.plot(ang, m)
    ax.axhline(0, c="gray", lw=0.5)
    ax.set_title(f"{P.field_of(v):.0f}/150 V/cm")
    ax.set_xlabel("angle from cathode (deg)")
axes[0].set_ylabel("dF, offset removed (%)")
fig.suptitle("5-us pairs, mean profile over the 4 cells of c2 and c9")
plt.show()
print(f"Plotted range: {lo:.1f} to {hi:.1f}%")

## Summary

| Claim | Outcome |
|---|---|
| fig5b-conductivity-calibration | not reproduced (10-fold rise gives 1.2 to 1.3%, not ~2%; 100-fold gives 11.8 to 11.9%) |
| fig1-pole-dimming | blocked, data missing |
| fig2a-fluovolt-saturation, fig2a-fluovolt-stable | blocked, data missing |
| fig2b-di8-climbs-to-570, fig2b-di8-bleaching, fig2b-di8-decline-within-set | blocked, data missing |
| fig2cd-stable-before-electroporation, fig2c-1000-within-set, fig2d-lasting-reduction | blocked, data missing |
| fig3-dye-sensitivity, fig3-fluovolt-nonlinear | blocked, data missing |
| fig4a-fluovolt-limits, fig4a-fluovolt-cathodic-decline | blocked, data missing |
| fig4b-di8-limits, fig4b-first-10-pulses, fig4b-cathodic-asymmetry, fig4c-equator-follows-cosine | blocked, data missing |
| fig4d-short-pulses-same-limit | blocked, data missing |
| fig5a-gd-extends-anodic, fig5a-gd-linear, fig5a-gd-cathodic, fig5a-gd-differences, fig5c-gd-reduces-conductivity | blocked, data missing |
| fig6a-ca-single-pulse, fig6b-ca-train-responders, fig6b-ca-650 | blocked, data missing |
| fig7a-cosine-profiles, fig7b-charging-during-pulse, fig7ab-test-pulse-recovery, fig7c-fast-conductivity-drop, fig7c-full-reseal-mild | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: the one tested claim is a computation that uses no data,
run as the claims file gives it. The variant measures of the dataset's own experiments place the
6-pixel trace automatically inside the outer edge of each cell, take the field direction from the
data, use every cell in a field of view, and collapse 20 and 40 frames in place of 40 and 60.

What was found about the dataset:

- It does not hold the data of any of the paper's figures. It holds a FluoVolt two-pulse experiment
  with intervals of 1 to 250 us and fields of 150 to 1570 V/cm, recorded with 1.00-mm electrodes in
  sets of 20 sham and 40 pulse frames, which the paper does not report (its Discussion names it as
  future work); its metadata names the paper as the publication of these data.
- The general description gives the intervals as 1, 5, 25 and 125 us; two fields of view are at 250 us.
- The pulse protocol of each set is only in free text; there is no stimulus or trial table. The
  20/40 frame order is confirmed by the data.
- The field direction, the pixel size and which cell of a field of view is the experiment's cell are
  not recorded; the 12 subjects are fields of view with 1 to 4 cells each.
- The listed field strengths (0.15 to 1.57 kV/cm for 50 to 250 V) are not proportional to voltage,
  and the files do not map amplitudes to fields.
- `starting_time` is the Unix time of the acquisition, not an offset from `session_start_time`.
- On the cells, the pulse frames are 9 to 17% dimmer than the sham frames even in sham sets, a
  uniform offset that the paper's measure does not remove.

In [ ]:
print(f"Notebook run time: {time.time() - started:.0f} s")